<a href="https://colab.research.google.com/github/LampOfSocrates/Surrey_EEEM071_Coursework/blob/main/EEEM071_CourseWork_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EEEM071 — Vehicle Re-ID Training

Runs in **Google Colab** (mounts Drive, clones repo, installs deps) and **locally** (uses repo working directory). Environment is detected automatically via `in_colab()`.

Parameterised with [Papermill](https://papermill.readthedocs.io). Pre-configured experiment notebooks live in `notebooks/`. Run the full LR sweep with:
```bash
python scripts/run_lr_sweep.py
```

In [ ]:
# ── Run mode ──────────────────────────────────────────────────────────────────
# Change this to switch between experiment modes. All other cells auto-adapt.
#
#   "single"          → train/eval one experiment (parameters cell below)
#   "lr_sweep"        → MobileNetV3-Small LR sweep (5 runs) + comparison
#   "all_experiments" → run all 8 experiment configs + comparison
#   "compare"         → compare existing results only, no training

run_mode = "single"

In [ ]:
# ── Student info ──────────────────────────────────────────────────────────────
student_id   = "your_student_id"
student_name = "Your_Name"

# ── Dataset ───────────────────────────────────────────────────────────────────
source_dataset = "veri"         # dataset used for training
target_dataset = "veri"         # dataset used for evaluation
data_root      = "../data/VeRi" # local path; overridden automatically in Colab

# ── Colab settings (ignored when running locally) ─────────────────────────────
colab_repo_url        = "https://github.com/LampOfSocrates/Surrey_EEEM071_Coursework.git"
colab_repo_dir        = "/content/Surrey_EEEM071_Coursework"
colab_drive_data_path = "/content/drive/MyDrive"

# ── Model ─────────────────────────────────────────────────────────────────────
arch          = "mobilenet_v3_small"  # resnet50 | resnet18 | mobilenet_v3_small | vgg16
no_pretrained = False                 # set True to skip ImageNet pretrained weights

# ── Image dimensions ──────────────────────────────────────────────────────────
image_height = 224
image_width  = 224

# ── Optimisation ──────────────────────────────────────────────────────────────
optimizer     = "amsgrad"  # adam | amsgrad | sgd | rmsprop
learning_rate = 0.0003
weight_decay  = 5e-4
max_epochs    = 1
stepsize      = "20 40"   # space-separated LR decay milestones
gamma         = 0.1       # LR decay factor
seed          = 1

# ── Batch sizes ───────────────────────────────────────────────────────────────
train_batch_size = 64
test_batch_size  = 100

# ── Loss ──────────────────────────────────────────────────────────────────────
margin      = 0.3  # triplet loss margin
lambda_xent = 1.0  # cross-entropy loss weight
lambda_htri = 1.0  # hard-triplet loss weight
label_smooth = False

# ── Augmentation ──────────────────────────────────────────────────────────────
random_erase = False
color_jitter = False
color_aug    = False

# ── Hardware & output ─────────────────────────────────────────────────────────
gpu_devices   = "0"                      # e.g. "0" or "0,1"
use_cpu       = False
eval_freq     = -1                       # -1 = evaluate only at end
evaluate_only = False                    # skip training, run eval only
save_dir      = "logs/mobilenet_v3_small-veri-lr3e-4"
data_fraction = 0.01                     # 0.01 = 1% for fast iteration; 1.0 = full dataset
compare_logs_dir = ""             # "" = scan all logs/, "latest" = newest run, or specific path

## 1. Environment Detection

In [2]:
import json
import os
import re
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def in_colab() -> bool:
    """Return True when running inside Google Colab."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IS_COLAB = in_colab()
print(f"Environment : {'Google Colab' if IS_COLAB else 'local'}")
print(f"Python      : {sys.executable}")

Environment : Google Colab
Python      : /usr/bin/python3


## 2a. Colab Setup

> **Colab checklist before running:**
> 1. *Edit → Notebook settings* → set Hardware accelerator to **GPU**
> 2. Click the folder icon on the left and **mount Google Drive** (or wait for the cell below to prompt you)
> 3. Upload `VeRi.zip` to `My Drive/` and note the path — set `colab_drive_data_path` in the parameters cell
>
> *Skipped automatically when not in Colab.*

In [ ]:
if IS_COLAB:
    # ── Fallback defaults (used if parameters cell hasn't run yet) ────────────
    colab_repo_url        = globals().get("colab_repo_url",        "https://github.com/LampOfSocrates/Surrey_EEEM071_Coursework.git")
    colab_repo_dir        = globals().get("colab_repo_dir",        "/content/Surrey_EEEM071_Coursework")
    colab_drive_data_path = globals().get("colab_drive_data_path", "/content/drive/MyDrive")

    import subprocess, os, sys
    from pathlib import Path

    # ── Bootstrap: clone if missing, then ALWAYS force-sync ──────────────────
    # Force-sync BEFORE importing from the repo — ensures scripts/colab_setup.py
    # exists even when the runtime has stale/old code on disk.
    if not Path(colab_repo_dir).exists():
        print("[colab_setup] Cloning repo for the first time ...")
        subprocess.run(["git", "clone", colab_repo_url, colab_repo_dir], check=True)
        print("[colab_setup] Clone complete.")
    else:
        print("[colab_setup] Repo present — force-syncing to origin/main ...")
        subprocess.run(["git", "fetch", "origin"], cwd=colab_repo_dir, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=colab_repo_dir, check=True)
        print("[colab_setup] Up to date with origin/main.")

    # ── Now safe to import from the repo ─────────────────────────────────────
    sys.path.insert(0, colab_repo_dir)
    from scripts.colab_setup import setup_colab

    data_root = setup_colab(
        repo_url=colab_repo_url,
        repo_dir=colab_repo_dir,
        drive_data_path=colab_drive_data_path,
    )

else:
    print("[colab_setup] Colab setup skipped (local environment)")


## 2b. Local Setup

*Skipped automatically when running in Colab.*

In [4]:
if not IS_COLAB:
    # Normalise working directory to the folder containing this notebook
    # so that main.py and src/ are importable regardless of where papermill
    # was invoked from.
    notebook_dir = Path(globals().get("__vsc_ipynb_file__", "")).parent
    if notebook_dir.exists() and notebook_dir != Path("."):
        os.chdir(notebook_dir)
    # Fallback: if this file is in notebooks/, step up one level to repo root
    if not Path("main.py").exists() and Path("../main.py").exists():
        os.chdir("..")
    print(f"Working directory: {Path.cwd()}")

    # GPU check (non-fatal)
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        if result.returncode == 0:
            print("\n".join(result.stdout.splitlines()[:10]))
        else:
            print("nvidia-smi not available — training will use CPU")
    except FileNotFoundError:
        print("nvidia-smi not found — training will use CPU")

else:
    print("Local setup skipped (Colab environment)")

print(f"\nStudent  : {student_name} ({student_id})")
print(f"Arch     : {arch}")
print(f"Dataset  : {source_dataset} -> {target_dataset}")
print(f"Data     : {data_root}")
print(f"Epochs   : {max_epochs}  |  LR: {learning_rate}  |  Optim: {optimizer}")

Local setup skipped (Colab environment)

Student  : Your_Name (your_student_id)
Arch     : mobilenet_v3_small
Dataset  : veri -> veri
Data     : /content/drive/MyDrive
Epochs   : 1  |  LR: 0.0003  |  Optim: amsgrad


## 3. Build Training Command

In [ ]:
if run_mode != "single":
    print(f"Skipping command build (run_mode={run_mode!r}). Commands are built inline in sections 6–7.")
else:
    cmd = [
        sys.executable, "main.py",
        "-s", source_dataset,
        "-t", target_dataset,
        "-a", arch,
        "--root",             data_root,
        "--height",           str(image_height),
        "--width",            str(image_width),
        "--optim",            optimizer,
        "--lr",               str(learning_rate),
        "--weight-decay",     str(weight_decay),
        "--max-epoch",        str(max_epochs),
        "--stepsize",         *stepsize.split(),
        "--gamma",            str(gamma),
        "--train-batch-size", str(train_batch_size),
        "--test-batch-size",  str(test_batch_size),
        "--margin",           str(margin),
        "--lambda-xent",      str(lambda_xent),
        "--lambda-htri",      str(lambda_htri),
        "--seed",             str(seed),
        "--eval-freq",        str(eval_freq),
        "--save-dir",         save_dir,
        "--gpu-devices",      gpu_devices,
        "--data-fraction",    str(data_fraction),
    ]

    if label_smooth:  cmd.append("--label-smooth")
    if random_erase:  cmd.append("--random-erase")
    if color_jitter:  cmd.append("--color-jitter")
    if color_aug:     cmd.append("--color-aug")
    if no_pretrained: cmd.append("--no-pretrained")
    if use_cpu:       cmd.append("--use-cpu")
    if evaluate_only: cmd.append("--evaluate")

    env = os.environ.copy()
    env["STUDENT_ID"]   = student_id
    env["STUDENT_NAME"] = student_name

    shell_cmd = (
        f"STUDENT_ID='{student_id}' STUDENT_NAME='{student_name}' "
        + " ".join(cmd)
    )
    print(f"Shell equivalent:\n{shell_cmd}")

## 4. Run Training

In [ ]:
if run_mode != "single":
    print(f"Skipping single-run training (run_mode={run_mode!r}). See sections 6–8 below.")
else:
    import argparse, time
    from main import run as run_experiment

    # Build an args Namespace directly — no subprocess, no sys.argv parsing
    _arg_list = [
        "-s", source_dataset, "-t", target_dataset,
        "-a", arch,
        "--root",             data_root,
        "--height",           str(image_height),
        "--width",            str(image_width),
        "--optim",            optimizer,
        "--lr",               str(learning_rate),
        "--weight-decay",     str(weight_decay),
        "--max-epoch",        str(max_epochs),
        "--stepsize",         *stepsize.split(),
        "--gamma",            str(gamma),
        "--train-batch-size", str(train_batch_size),
        "--test-batch-size",  str(test_batch_size),
        "--margin",           str(margin),
        "--lambda-xent",      str(lambda_xent),
        "--lambda-htri",      str(lambda_htri),
        "--seed",             str(seed),
        "--eval-freq",        str(eval_freq),
        "--save-dir",         save_dir,
        "--gpu-devices",      gpu_devices,
        "--data-fraction",    str(data_fraction),
    ]
    if label_smooth:  _arg_list.append("--label-smooth")
    if random_erase:  _arg_list.append("--random-erase")
    if color_jitter:  _arg_list.append("--color-jitter")
    if color_aug:     _arg_list.append("--color-aug")
    if no_pretrained: _arg_list.append("--no-pretrained")
    if use_cpu:       _arg_list.append("--use-cpu")
    if evaluate_only: _arg_list.append("--evaluate")

    from args import argument_parser
    _parsed_args = argument_parser().parse_args(_arg_list)

    os.environ["STUDENT_ID"]   = student_id
    os.environ["STUDENT_NAME"] = student_name

    t0 = time.time()
    run_experiment(_parsed_args, redirect_stdout=False)  # output goes to notebook cell
    elapsed = time.time() - t0
    print(f"\nFinished in {elapsed:.1f}s")

## 5. Metrics Visualisation

Parse training logs and evaluation results, then plot training curves and CMC metrics.

In [7]:
if run_mode != "single":
    print(f"Skipping single-run metrics (run_mode={run_mode!r}).")
else:
    from src.utils.metrics_viz import render_metrics

    _cfg = {
        "arch": arch, "learning_rate": learning_rate, "max_epochs": max_epochs,
        "optimizer": optimizer, "train_batch_size": train_batch_size,
        "image_height": image_height, "image_width": image_width,
        "label_smooth": label_smooth, "random_erase": random_erase,
        "color_jitter": color_jitter, "margin": margin, "save_dir": save_dir,
        "evaluate_only": evaluate_only,
    }
    _metrics = render_metrics(_output_lines, _cfg, elapsed=elapsed)

Parsed 3 batch log entries
mAP       : 10.3%
Rank-1    : 0.0%
Rank-5    : 33.3%
Rank-10   : 33.3%
Rank-20   : 77.8%
Saved: logs/mobilenet_v3_small-veri-lr3e-4/training_curves.png
Saved: logs/mobilenet_v3_small-veri-lr3e-4/eval_metrics.png

Metrics JSON -> logs/mobilenet_v3_small-veri-lr3e-4/metrics.json


## 6. Run LR Sweep

Runs all 5 MobileNetV3-Small learning-rate variants (3e-3 → 3e-5) via Papermill,
exports each to HTML, then runs the comparison notebook.

> Set `run_mode = "lr_sweep"` in the parameters cell to activate this section.

In [ ]:
if run_mode != "lr_sweep":
    print(f"Skipping LR sweep (run_mode={run_mode!r}). Set run_mode='lr_sweep' to run.")
else:
    import time
    from src.utils.metrics_viz import render_metrics

    # ── LR sweep configurations (only lr and save_dir vary) ──────────────────
    LR_SWEEP = [
        {"learning_rate": 3e-3,  "save_dir": "logs/mobilenet_v3_small-veri-lr3e-3"},
        {"learning_rate": 1e-3,  "save_dir": "logs/mobilenet_v3_small-veri-lr1e-3"},
        {"learning_rate": 3e-4,  "save_dir": "logs/mobilenet_v3_small-veri-lr3e-4"},
        {"learning_rate": 1e-4,  "save_dir": "logs/mobilenet_v3_small-veri-lr1e-4"},
        {"learning_rate": 3e-5,  "save_dir": "logs/mobilenet_v3_small-veri-lr3e-5"},
    ]

    def _build_cmd(lr, run_save_dir):
        """Build main.py command for one LR sweep run."""
        c = [
            sys.executable, "main.py",
            "-s", source_dataset, "-t", target_dataset,
            "-a", "mobilenet_v3_small",
            "--root",             data_root,
            "--height",           "224",
            "--width",            "224",
            "--optim",            "amsgrad",
            "--lr",               str(lr),
            "--weight-decay",     str(weight_decay),
            "--max-epoch",        str(max_epochs),
            "--stepsize",         *stepsize.split(),
            "--gamma",            str(gamma),
            "--train-batch-size", "64",
            "--test-batch-size",  str(test_batch_size),
            "--margin",           str(margin),
            "--lambda-xent",      str(lambda_xent),
            "--lambda-htri",      str(lambda_htri),
            "--seed",             str(seed),
            "--eval-freq",        str(eval_freq),
            "--save-dir",         run_save_dir,
            "--gpu-devices",      gpu_devices,
            "--data-fraction",    str(data_fraction),
        ]
        if use_cpu: c.append("--use-cpu")
        return c

    sweep_results = []

    for i, cfg_override in enumerate(LR_SWEEP, 1):
        lr          = cfg_override["learning_rate"]
        run_save_dir = cfg_override["save_dir"]

        print(f"\n{'='*65}")
        print(f"  Run {i}/{len(LR_SWEEP)} — lr={lr}  save_dir={run_save_dir}")
        print(f"{'='*65}\n")

        run_cmd = _build_cmd(lr, run_save_dir)
        output_lines = []
        t0 = time.time()

        with subprocess.Popen(
            run_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env,
        ) as proc:
            for line in proc.stdout:
                print(line, end="")
                output_lines.append(line)
            proc.wait()

        elapsed = time.time() - t0
        if proc.returncode != 0:
            print(f"\nERROR: run exited with code {proc.returncode}")
            sweep_results.append(None)
            continue

        # Render per-run metrics and plots
        run_cfg = {
            "arch": "mobilenet_v3_small", "learning_rate": lr,
            "max_epochs": max_epochs, "optimizer": "amsgrad",
            "train_batch_size": 64, "image_height": 224, "image_width": 224,
            "label_smooth": False, "random_erase": False,
            "color_jitter": False, "margin": margin, "save_dir": run_save_dir,
            "evaluate_only": False,
        }
        metrics = render_metrics(output_lines, run_cfg, elapsed=elapsed)
        sweep_results.append(metrics)

    print(f"\n{'='*65}")
    print(f"  LR sweep complete — {sum(r is not None for r in sweep_results)}/{len(LR_SWEEP)} runs succeeded")
    print(f"{'='*65}")

## 7. Run ALL Experiments

Generates all 9 experiment notebooks (LR sweep + ResNet-50 + ResNet-18 + SGD/smooth + eval-only)
and runs each one via Papermill sequentially.

> Set `run_mode = "all_experiments"` in the parameters cell to activate this section.

In [ ]:
if run_mode != "all_experiments":
    print(f"Skipping all-experiments run (run_mode={run_mode!r}). Set run_mode='all_experiments' to run.")
else:
    import time
    from src.utils.metrics_viz import render_metrics

    # All experiment configurations (mirrors generate_experiment_notebooks.py)
    # 5 models x 3 LR variants (low / baseline / high) = 15 runs, 1 epoch each
    # data_fraction=0.005 → 0.5% of dataset for fast iteration
    ALL_EXPERIMENTS = [
        # ── MobileNetV3-Small (amsgrad, 224×224) ──────────────────────────────
        {"name": "mobilenet_v3_small_lr_low",      "arch": "mobilenet_v3_small", "max_epochs": 1, "lr": 1e-4, "optim": "amsgrad", "height": 224, "width": 224, "bs": 64, "data_fraction": 0.005, "save_dir": "logs/mobilenet_v3_small-lr1e-4"},
        {"name": "mobilenet_v3_small_lr_base",     "arch": "mobilenet_v3_small", "max_epochs": 1, "lr": 3e-4, "optim": "amsgrad", "height": 224, "width": 224, "bs": 64, "data_fraction": 0.005, "save_dir": "logs/mobilenet_v3_small-lr3e-4"},
        {"name": "mobilenet_v3_small_lr_high",     "arch": "mobilenet_v3_small", "max_epochs": 1, "lr": 1e-3, "optim": "amsgrad", "height": 224, "width": 224, "bs": 64, "data_fraction": 0.005, "save_dir": "logs/mobilenet_v3_small-lr1e-3"},
        # ── ResNet-18 (adam, 256×128) ─────────────────────────────────────────
        {"name": "resnet18_lr_low",                "arch": "resnet18",           "max_epochs": 1, "lr": 1e-4, "optim": "adam",    "height": 256, "width": 128, "bs": 64, "data_fraction": 0.005, "save_dir": "logs/resnet18-lr1e-4"},
        {"name": "resnet18_lr_base",               "arch": "resnet18",           "max_epochs": 1, "lr": 3e-4, "optim": "adam",    "height": 256, "width": 128, "bs": 64, "data_fraction": 0.005, "save_dir": "logs/resnet18-lr3e-4"},
        {"name": "resnet18_lr_high",               "arch": "resnet18",           "max_epochs": 1, "lr": 1e-3, "optim": "adam",    "height": 256, "width": 128, "bs": 64, "data_fraction": 0.005, "save_dir": "logs/resnet18-lr1e-3"},
        # ── ResNet-50 / amsgrad (256×128) ────────────────────────────────────
        {"name": "resnet50_lr_low",                "arch": "resnet50",           "max_epochs": 1, "lr": 1e-4, "optim": "amsgrad", "height": 256, "width": 128, "bs": 32, "data_fraction": 0.005, "save_dir": "logs/resnet50-lr1e-4"},
        {"name": "resnet50_lr_base",               "arch": "resnet50",           "max_epochs": 1, "lr": 3e-4, "optim": "amsgrad", "height": 256, "width": 128, "bs": 32, "data_fraction": 0.005, "save_dir": "logs/resnet50-lr3e-4"},
        {"name": "resnet50_lr_high",               "arch": "resnet50",           "max_epochs": 1, "lr": 1e-3, "optim": "amsgrad", "height": 256, "width": 128, "bs": 32, "data_fraction": 0.005, "save_dir": "logs/resnet50-lr1e-3"},
        # ── ResNet-50 / SGD + label-smooth (256×128) ─────────────────────────
        {"name": "resnet50_sgd_lr_low",            "arch": "resnet50",           "max_epochs": 1, "lr": 1e-3, "optim": "sgd",     "height": 256, "width": 128, "bs": 32, "data_fraction": 0.005, "save_dir": "logs/resnet50-sgd-lr1e-3", "label_smooth": True, "margin": 0.5},
        {"name": "resnet50_sgd_lr_base",           "arch": "resnet50",           "max_epochs": 1, "lr": 1e-2, "optim": "sgd",     "height": 256, "width": 128, "bs": 32, "data_fraction": 0.005, "save_dir": "logs/resnet50-sgd-lr1e-2", "label_smooth": True, "margin": 0.5},
        {"name": "resnet50_sgd_lr_high",           "arch": "resnet50",           "max_epochs": 1, "lr": 1e-1, "optim": "sgd",     "height": 256, "width": 128, "bs": 32, "data_fraction": 0.005, "save_dir": "logs/resnet50-sgd-lr1e-1", "label_smooth": True, "margin": 0.5},
        # ── VGG-16 (amsgrad, 224×224) ─────────────────────────────────────────
        {"name": "vgg16_lr_low",                   "arch": "vgg16",              "max_epochs": 1, "lr": 1e-4, "optim": "amsgrad", "height": 224, "width": 224, "bs": 32, "data_fraction": 0.005, "save_dir": "logs/vgg16-lr1e-4"},
        {"name": "vgg16_lr_base",                  "arch": "vgg16",              "max_epochs": 1, "lr": 3e-4, "optim": "amsgrad", "height": 224, "width": 224, "bs": 32, "data_fraction": 0.005, "save_dir": "logs/vgg16-lr3e-4"},
        {"name": "vgg16_lr_high",                  "arch": "vgg16",              "max_epochs": 1, "lr": 1e-3, "optim": "amsgrad", "height": 224, "width": 224, "bs": 32, "data_fraction": 0.005, "save_dir": "logs/vgg16-lr1e-3"},
    ]

    def _build_exp_cmd(exp):
        c = [
            sys.executable, "main.py",
            "-s", source_dataset, "-t", target_dataset,
            "-a", exp["arch"],
            "--root",             data_root,
            "--height",           str(exp["height"]),
            "--width",            str(exp["width"]),
            "--optim",            exp["optim"],
            "--lr",               str(exp["lr"]),
            "--weight-decay",     str(weight_decay),
            "--max-epoch",        str(exp["max_epochs"]),
            "--stepsize",         *exp.get("stepsize", stepsize).split(),
            "--gamma",            str(gamma),
            "--train-batch-size", str(exp["bs"]),
            "--test-batch-size",  str(test_batch_size),
            "--margin",           str(exp.get("margin", margin)),
            "--lambda-xent",      str(lambda_xent),
            "--lambda-htri",      str(lambda_htri),
            "--seed",             str(seed),
            "--eval-freq",        str(eval_freq),
            "--save-dir",         exp["save_dir"],
            "--gpu-devices",      gpu_devices,
            "--data-fraction",    str(exp.get("data_fraction", data_fraction)),
        ]
        if exp.get("label_smooth"): c.append("--label-smooth")
        if exp.get("random_erase"): c.append("--random-erase")
        if exp.get("color_jitter"): c.append("--color-jitter")
        if use_cpu: c.append("--use-cpu")
        return c

    completed, failed = [], []

    for i, exp in enumerate(ALL_EXPERIMENTS, 1):
        print(f"\n{'='*65}")
        print(f"  Experiment {i}/{len(ALL_EXPERIMENTS)}: {exp['name']}")
        print(f"  arch={exp['arch']}  lr={exp['lr']}  epochs={exp['max_epochs']}")
        print(f"{'='*65}\n")

        run_cmd = _build_exp_cmd(exp)
        output_lines = []
        t0 = time.time()

        with subprocess.Popen(
            run_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env,
        ) as proc:
            for line in proc.stdout:
                print(line, end="")
                output_lines.append(line)
            proc.wait()

        elapsed = time.time() - t0
        if proc.returncode != 0:
            print(f"\nERROR: experiment exited with code {proc.returncode}")
            failed.append(exp["name"])
            continue

        run_cfg = {
            "arch": exp["arch"], "learning_rate": exp["lr"],
            "max_epochs": exp["max_epochs"], "optimizer": exp["optim"],
            "train_batch_size": exp["bs"],
            "image_height": exp["height"], "image_width": exp["width"],
            "label_smooth": exp.get("label_smooth", False),
            "random_erase": exp.get("random_erase", False),
            "color_jitter": exp.get("color_jitter", False),
            "margin": exp.get("margin", margin),
            "save_dir": exp["save_dir"], "evaluate_only": False,
        }
        render_metrics(output_lines, run_cfg, elapsed=elapsed)
        completed.append(exp["name"])

    print(f"\n{'='*65}")
    print(f"  All experiments done — {len(completed)} succeeded, {len(failed)} failed")
    if failed:
        print(f"  Failed: {', '.join(failed)}")
    print(f"{'='*65}")

## 8. Compare Results

Scans all completed runs under `logs/`, aggregates metrics, and produces a leaderboard
table (CSV / JSON / Markdown) plus a metric glossary explaining each score.

> Runs automatically after `lr_sweep` and `all_experiments` modes.
> Set `run_mode = "compare"` to run standalone against existing logs.

In [ ]:
if run_mode not in ("compare", "lr_sweep", "all_experiments"):
    print(f"Skipping comparison (run_mode={run_mode!r}). Set run_mode='compare' to run standalone.")
else:
    import re as _re
    import sys
    from pathlib import Path
    from IPython.display import display, Markdown

    sys.path.insert(0, str(Path(".").resolve()))
    from tools.summarize_experiments import load_experiment
    from src.utils.compare_viz import render_all_comparisons

    # ── Resolve which logs folder to scan ─────────────────────────────────────
    _logs_spec = globals().get("compare_logs_dir", "").strip()
    _TS_RE     = _re.compile(r'\d{8}_\d{4}$')

    if _logs_spec == "latest":
        _candidates = sorted(
            [d for d in Path("logs").iterdir() if d.is_dir() and _TS_RE.search(d.name)],
            key=lambda p: p.name,
        )
        if _candidates:
            # latest run's parent — scan from there so we compare all runs up to latest
            logs_root = Path("logs")
            print(f"[compare] Latest run: {_candidates[-1].name}  (scanning all of logs/)")
        else:
            logs_root = Path("logs")
            print("[compare] No timestamped runs found — scanning logs/")
    elif _logs_spec:
        logs_root = Path(_logs_spec)
        print(f"[compare] Scanning specified path: {logs_root}")
    else:
        logs_root = Path("logs")
        print(f"[compare] Scanning all of {logs_root}/")

    out_dir = Path("outputs/comparison")
    out_dir.mkdir(parents=True, exist_ok=True)

    summary_files = sorted(logs_root.rglob("summary.json"))
    if not summary_files:
        print(f"No summary.json files found under {logs_root!r}. Run at least one experiment first.")
    else:
        print(f"Found {len(summary_files)} completed run(s):")
        for f in summary_files:
            print(f"  {f}")

        rows = [r for f in summary_files if (r := load_experiment(f)) is not None]
        rows.sort(
            key=lambda r: (float(r["best_val_mAP"] or 0), float(r["best_val_rank1"] or 0)),
            reverse=True,
        )

        render_all_comparisons(rows, out_dir=str(out_dir))


## 6. Output Files

Checkpoints, logs, plots and `metrics.json` are all written to `save_dir`.
The HTML export is generated by `scripts/run_lr_sweep.py` after Papermill finishes.

In [15]:
log_dir = Path(save_dir)
if log_dir.exists():
    files = sorted(log_dir.rglob("*"))
    print(f"Output files in {log_dir}:")
    for f in files:
        if f.is_file():
            print(f"  {str(f.relative_to(log_dir)):<40}  {f.stat().st_size:>10,} bytes")
else:
    print(f"Log directory not yet created: {log_dir}")

print(f"\nRun summary:")
print(f"  arch={arch}, lr={learning_rate}, epochs={max_epochs}, optim={optimizer}, seed={seed}")
print(f"  image={image_height}x{image_width}, batch={train_batch_size}")
print(f"  aug: random_erase={random_erase}, color_jitter={color_jitter}")
print(f"  loss: margin={margin}, label_smooth={label_smooth}")

Output files in logs/mobilenet_v3_small-veri-lr3e-4:
  best_model.pth.tar                        20,447,819 bytes
  config.json                                    1,171 bytes
  eval_metrics.png                              55,150 bytes
  log_train.txt                                  7,093 bytes
  metrics.csv                                      447 bytes
  metrics.json                                     834 bytes
  model.pth.tar-1                           20,447,819 bytes
  plots/grad_norm_vs_epoch.png                  20,887 bytes
  plots/id_loss_vs_epoch.png                    19,769 bytes
  plots/loss_vs_epoch.png                       21,115 bytes
  plots/lr_vs_epoch.png                         29,437 bytes
  plots/triplet_loss_vs_epoch.png               20,202 bytes
  plots/val_map_vs_epoch.png                    20,090 bytes
  plots/val_minp_vs_epoch.png                   20,463 bytes
  plots/val_rank1_vs_epoch.png                  18,675 bytes
  retrieval_metrics.json        